<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# Clasificación de Sentimientos con BERT

In [ ]:
!pip install datasets

Importamos AutoTokenizer, AutoModelForSequenceClassification y pipeline de Transformers

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

Cargamos el modelo preentrenado distilbert-base-uncased-finetuned-sst-2-english y su tokenizer

In [ ]:
# Cargar el modelo preentrenado y el tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Cargamos el dataset de IMDB para clasificar sentimientos

Usamos solo los 10 primeros textos

In [ ]:
dataset = load_dataset("imdb")
texts = dataset['train']['text'][:10]

Tokenizamos los textos

In [ ]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

Realizamos predicciones

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

Obtenemos los resultados

In [ ]:
labels = ['negative', 'positive']
for i, text in enumerate(texts):
    pred_label = labels[predictions[i].argmax()]
    confidence = predictions[i].max().item()
    print(f"Text: {text}")
    print(f"Prediction: {pred_label} (Confidence: {confidence:.4f})")
    print()

Realizamos lo mismo usando pipeline de transformer

In [ ]:
clasifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, truncation=True)
pipeline_results = clasifier(texts)

Resultados del pipeline

In [ ]:
for i, (text, result) in enumerate(zip(texts, pipeline_results)):
    print(f"Texto {i+1}: {text[:100]}...")
    print(f"Predicción: {result['label']} (Confianza: {result['score']:.4f})")

# NER Tagging

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

nlp = pipeline("ner", model=model, tokenizer=tokenizer)
example = "My name is Wolfgang Mozart and I live in Berlin"

ner_results = nlp(example)
print(ner_results)


B-PER', 'score': np.float32(0.9990833), 'index': 4, 'word': 'Wolfgang', 'start': 11, 'end': 19}, {'entity': 'I-PER', 'score': np.float32(0.9967392), 'index': 5, 'word': 'Mozart', 'start': 20, 'end': 26}, {'entity': 'B-LOC', 'score': np.float32(0.99963164), 'index': 10, 'word': 'Berlin', 'start': 41, 'end': 47}]


